In [8]:
%load_ext autoreload
%autoreload 2

In [9]:
import pandas as pd
import numpy as np

In [41]:
print('test')

test


# Generate KG

In [ ]:
!python construct_kg.py

/mnt/danderson/home/pander14/GitHub/kacd/construct_kg.py:17: LangChainDeprecationWarning: The class `Neo4jGraph` was deprecated in LangChain 0.3.8 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-neo4j package and should be used instead. To use it run `pip install -U :class:`~langchain-neo4j` and import as `from :class:`~langchain_neo4j import Neo4jGraph``.
  graph = Neo4jGraph(NEO4J_URI,NEO4J_USERNAME,NEO4J_PASSWORD,NEO4J_DATABASE, refresh_schema=False)
../clbp_causal_md_24jul26/jarvik2014_clean.md
../clbp_causal_md_24jul26/lee2015_clean.md
../clbp_causal_md_24jul26/assari_clean.md
../clbp_causal_md_24jul26/kohler2018_clean.md
../clbp_causal_md_24jul26/morin2020_clean.md
../clbp_causal_md_24jul26/amiri_clean.md
../clbp_causal_md_24jul26/böckerman_clean.md
../clbp_causal_md_24jul26/zhao2022_clean.md
../clbp_causal_md_24jul26/miloyan2019_clean.md
../clbp_causal_md_24jul26/mulia2022_clean.md
../clbp_causal_md_24jul26/bowen_clean.md
../clbp_cau

# Generate Predictions

In [ ]:
!python query_kg.py

/mnt/danderson/home/pander14/GitHub/kacd/context_construction/build_context.py:12: LangChainDeprecationWarning: The class `Neo4jGraph` was deprecated in LangChain 0.3.8 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-neo4j package and should be used instead. To use it run `pip install -U :class:`~langchain-neo4j` and import as `from :class:`~langchain_neo4j import Neo4jGraph``.
  graph = Neo4jGraph(NEO4J_URI,NEO4J_USERNAME,NEO4J_PASSWORD,NEO4J_DATABASE, refresh_schema=False)
/mnt/danderson/home/pander14/venvs/kacd/lib/python3.10/site-packages/langchain_community/graphs/neo4j_graph.py:404: PreviewWarning: notifications_disabled_classifications is a preview feature. It might be changed without following the deprecation policy. See also https://github.com/neo4j/neo4j-python-driver/wiki/preview-features.
  self._driver = neo4j.GraphDatabase.driver(
/mnt/danderson/home/pander14/GitHub/kacd/context_construction/retriever_kgrag.py:43: LangChainDepr

In [ ]:
!python query_llm.py

In [ ]:
!python query_rag.py

In [ ]:
!python query_causal_lit_kg.py

In [ ]:
!python query_causal_lit_llm.py

In [ ]:
!python query_causal_lit_rag.py

# Combine all results

The next code blocks combine the predictions using plausibility, association, and temporality with the predictions using a prompt pulled from the literature "full_causal_literature".

In [ ]:
from config import RESULT_DIR

kgrag = pd.read_csv(RESULT_DIR / "kgrag.csv", index_col=0).drop(columns=["Label"])
llm = pd.read_csv(RESULT_DIR / "llm.csv", index_col=0).drop(columns=["Label"])
rag = pd.read_csv(RESULT_DIR / "rag.csv", index_col=0).drop(columns=["Label"])

kgrag_cl = pd.read_csv(RESULT_DIR / "kg+rag_full_causal_literature.csv", index_col=0).drop(columns=["Label"])
llm_cl = pd.read_csv(RESULT_DIR / "llm_full_causal_literature.csv", index_col=0).drop(columns=["Label"])
llm_cl["Causal Literature Report"] = '' # llm doesn't have a report, leaving it blank for consistency
rag_cl = pd.read_csv(RESULT_DIR / "llm+rag_full_causal_literature.csv", index_col=0).drop(columns=["Label"])

In [ ]:
from util.helpers import consolidate_causal_literature

# Apply to all causal literature dataframes
kgrag_cl = consolidate_causal_literature(kgrag_cl)
llm_cl = consolidate_causal_literature(llm_cl)
rag_cl = consolidate_causal_literature(rag_cl)

In [ ]:
kgrag = pd.merge(kgrag, kgrag_cl, on=["Var1", "Var2"])
rag = pd.merge(rag, rag_cl, on=["Var1", "Var2"])
llm = pd.merge(llm, llm_cl, on=["Var1", "Var2"])

In [ ]:
from config import RESULT_DIR

# Add context column to each dataframe
kgrag['context'] = 'kg+llm_full.csv'
llm['context'] = 'llm_full.csv'
rag['context'] = 'rag_full.csv'

# Concatenate all dataframes
consolidated = pd.concat([kgrag, llm, rag], ignore_index=True)

consolidated.to_csv(RESULT_DIR / "results_undirected_combined.csv")

# Generate directed results

This next step resolves directions between the knowledge systems. `with_proto` also leverages the proto model (a datadriven model) to help resolve bidirectional edges

In [ ]:
!python query_directions_without_proto.py

In [ ]:
!python query_directions_with_proto.py